# Stage 4 · Score the completed dense Lego baseline
Select **T4 ×2** in a fresh Kaggle notebook. This stage uses one GPU. Attach three private inputs: `stage0-input.zip`, `lego-dense-vanilla-s42-step50000.zip` and `stage4-lpips.zip`. Kaggle can also mount their extracted contents. Run the cells in order. The output is a review ZIP with held-out scores and images.

The frozen split selects every eighth image from the 200 published Lego test views. The evaluator renders all 25 selected cameras at 400 × 400 with a white background. Its progress messages estimate remaining time from this session’s measured seconds per view.

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import math
import os
import shutil
import statistics
import subprocess
import sys
import tempfile
import time
import zipfile

import torch
from IPython.display import HTML, display
from PIL import Image

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
work = outputs / "recovery-stage4-dense-eval"
relative_run = Path("runs/lego/dense-vanilla/seed-42")
run = work / relative_run
manifest = work / "manifests/lego.json"
review_zip = outputs / "lego-dense-vanilla-s42-eval.zip"
run_id = "08ff8d2fceda43759554aa6470054853"
checkpoint_hash = "a77230d4e17b3769445cf05e26b2bda7f094bf078418ff7d6366d906e19da8be"
data_hash = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
expected_ids = [f"test/r_{i}.png" for i in range(0, 200, 8)]

def sha(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def mounted(marker, zip_name):
    archives = list(inputs.rglob(zip_name))
    if not archives:
        roots = []
        for path in inputs.rglob(marker):
            if marker == "stage3-review.json":
                review = read_json(path)
                if (review.get("run_id"), review.get("checkpoint_step"), review.get("checkpoint_sha256")) != (run_id, 50_000, checkpoint_hash):
                    continue
            roots.append(path.parent)
        if len(roots) == 1:
            return roots[0]
    if len(archives) != 1:
        raise ValueError(f"Attach one {zip_name} or its matching expanded folder with {marker}.")
    root = Path(tempfile.mkdtemp(prefix="nerf-stage4-"))
    with zipfile.ZipFile(archives[0]) as archive:
        if marker not in archive.namelist() or any(
            not (root / name).resolve().is_relative_to(root) for name in archive.namelist()
        ):
            raise ValueError(f"Unexpected archive layout in {archives[0]}")
        archive.extractall(root)
    return root

def check_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle T4 accelerator and restart this notebook.")
    devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print(f"Evaluation device: cuda:0 ({devices[0]}).")
    return dict(torch=str(torch.__version__), cuda=torch.version.cuda, device="cuda:0", devices=devices)

hardware = check_gpu()
repo = mounted("RECOVERY_STAGE0.txt", "stage0-input.zip")
snapshot = mounted("stage3-review.json", "lego-dense-vanilla-s42-step50000.zip")
metric_kit = mounted("RECOVERY_LPIPS.txt", "stage4-lpips.zip")
run.mkdir(parents=True, exist_ok=True)

In [ ]:
sys.path.insert(0, str(repo))
from recovery.training import implementation_hash, load_state, settings

source_run = snapshot / relative_run
meta = read_json(source_run / "run.json")
stage3 = read_json(snapshot / "stage3-review.json")
assert stage3["complete"] is True and stage3["checkpoint_step"] == 50_000
assert stage3["run_id"] == meta["run_id"] == run_id
assert stage3["checkpoint_sha256"] == sha(source_run / "checkpoint.pt") == checkpoint_hash
assert meta["protocol"] == "dense-vanilla" and meta["scene"] == "lego" and meta["seed"] == 42
assert meta["train_views"] == 100 and meta["downscale"] == 2 and meta["data_hash"] == data_hash
assert implementation_hash() == meta["training_config"]["implementation_hash"] == code_hash
score_sources = {
    "recovery/media.py": "fd13945accd1ec6938b7ebceccec31fe1f8716dc2a2ad51d84068d560e09a3af",
    "sparse_nerf/evaluate.py": "eb2a0c14ab7aac4de41fc5de9e2e2316ba902e08a14cc2b1a06d7616171e58ca",
}
for name, expected in score_sources.items():
    assert hashlib.sha256((repo / name).read_text(encoding="utf-8").encode()).hexdigest() == expected
subprocess.run([sys.executable, "-m", "recovery.data", "--verify", str(snapshot / "manifests/lego.json"),
                "--source", str(repo / "data/blender/lego"), "--output", str(manifest)], cwd=repo, check=True)
frozen = read_json(manifest)
assert frozen["data_hash"] == data_hash and meta["training_config"] == settings("main", frozen["camera"])
assert [frozen["frames"][i]["file_path"] for i in frozen["splits"]["test"]] == expected_ids
assert meta["training_config"]["steps"] == 50_000 and load_state(source_run, meta)["step"] == 50_000
for filename in ("run.json", "checkpoint.pt"):
    destination = run / filename
    if destination.exists() and sha(destination) != sha(source_run / filename):
        raise ValueError(f"Existing {destination} belongs to different inputs. Start a fresh session.")
    if not destination.exists():
        shutil.copyfile(source_run / filename, destination)
assert sha(run / "checkpoint.pt") == checkpoint_hash
print(f"Verified step 50,000, run {run_id}, 100 training cameras and {len(expected_ids)} held-out cameras.")

In [ ]:
# Install the pinned LPIPS wheel in a stage-local directory. Kaggle supplies torch and torchvision.
wheel = metric_kit / "wheels/lpips-0.1.4-py3-none-any.whl"
torch_cache = metric_kit / "torch-cache"
weights = torch_cache / "hub/checkpoints/alexnet-owt-7be5be79.pth"
wheel_hash = sha(wheel)
weight_hash = sha(weights)
assert wheel_hash == "fd537af5828b69d2e6ffc0a397bd506dbc28ca183543617690844c08e102ec5e"
assert weight_hash == "7be5be791159472b1fbf3c69796f7cb30dca7ad8466c2df70058c37116cdee02"
site = work / "metric-packages"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--upgrade",
                "--target", str(site), str(wheel)], check=True)
env = os.environ.copy()
env.update(TORCH_HOME=str(torch_cache), PYTHONPATH=os.pathsep.join([str(site), str(repo), env.get("PYTHONPATH", "")]))
probe = """
import importlib.metadata, json, torch, torchvision
from sparse_nerf.evaluate import lpips_model
from recovery.media import model_fingerprint
net = lpips_model().eval()
with torch.inference_mode():
    sample = torch.zeros(1, 3, 64, 64)
    identical = net(sample, sample).item()
    different = net(sample, sample + 1).item()
assert identical == 0.0 and different > 0 and torch.isfinite(torch.tensor(different))
print(json.dumps(dict(lpips=importlib.metadata.version('lpips'), torchvision=torchvision.__version__,
                      lpips_model_sha256=model_fingerprint(net), identical=identical, different=different)))
"""
probe_output = subprocess.check_output([sys.executable, "-c", probe], cwd=repo, env=env, text=True)
lpips_info = json.loads(probe_output.strip().splitlines()[-1])
assert lpips_info["lpips"] == "0.1.4"
assert lpips_info["lpips_model_sha256"] == "913e6288d265ca661f26fe0426b840bc69bafcf9fcda0fb2c99586891562ea9e"
print("Offline LPIPS-Alex v0.1 ready.", lpips_info)

In [ ]:
metrics_path = run / "eval/metrics.json"
chunk = 1024
def score():
    command = [sys.executable, "-u", "-m", "recovery.media", "eval", "--run", str(run),
               "--manifest", str(manifest), "--device", "cuda:0", "--chunk", str(chunk)]
    log_path = work / "evaluation.log"
    first_view_time = None
    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
        command, cwd=repo, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                if line.startswith("[test "):
                    number = int(line.split("/", 1)[0].split()[-1])
                    if number == 1:
                        first_view_time = time.monotonic()
                    elif first_view_time is not None:
                        seconds = (time.monotonic() - first_view_time) / (number - 1)
                        print(f"Measured {seconds:.1f} s/view; about {(25-number)*seconds/60:.1f} min left.", flush=True)
            if process.wait():
                raise RuntimeError(f"Evaluation exited with status {process.returncode}. See {log_path}.")
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
if metrics_path.exists():
    print("Found completed metrics in this session. Checking them against the inputs below.")
else:
    score()

In [ ]:
from recovery.media import digest

result = read_json(metrics_path)
compare = result["comparison"]
rows = result["frames"]
assert result["run_id"] == run_id and result["train_views"] == 100
assert (result["scene"], result["protocol"], result["seed"]) == ("lego", "dense-vanilla", 42)
assert result["model_sha256"] == "3f137f5bc805121adf4b4e9eef1865da227559854d17fc5a531d951efbc33de8"
assert compare["data_hash"] == data_hash and compare["training_config"] == meta["training_config"]
assert compare["lpips_weights_sha256"] == lpips_info["lpips_model_sha256"]
assert compare["test_ids"] == [r["frame"] for r in rows] == expected_ids
assert (compare["width"], compare["height"], compare["white_background"]) == (400, 400, True)
assert result["comparison_hash"] == digest(compare)
assert math.isfinite(result["test_seconds"]) and result["test_seconds"] > 0
for key in ("psnr", "ssim", "lpips", "white_psnr", "foreground_psnr", "white_foreground_psnr", "foreground_opacity"):
    values = [r[key] for r in rows]
    assert all(map(math.isfinite, values)) and math.isclose(statistics.mean(values), result["mean"][key], abs_tol=1e-8)
assert math.isclose(result["mean"]["white_psnr"], 9.492596382193124, abs_tol=1e-5)

ranked = sorted(rows, key=lambda r: (r["psnr"], r["frame"]))
rank_examples = {label: dict(frame=r["frame"], psnr=r["psnr"]) for label, r in
                 zip(("lowest", "median", "highest"), (ranked[0], ranked[12], ranked[-1]))}
files = {"eval/metrics.json": metrics_path}
if (work / "evaluation.log").is_file():
    files["evaluation.log"] = work / "evaluation.log"
for index in (0, 12, 24):
    for kind in ("rgb", "target", "depth", "opacity"):
        name = f"view-{index:03d}-{kind}.png"
        assert rows[index]["artifacts"][kind] == name
        path = metrics_path.parent / name
        with Image.open(path) as picture:
            assert picture.format == "PNG" and picture.size == (400, 400)
            picture.verify()
        files[f"eval/{name}"] = path

review = dict(run_id=run_id, checkpoint_step=50_000, checkpoint_sha256=checkpoint_hash,
              model_sha256=result["model_sha256"], data_hash=data_hash, training_code_sha256=code_hash,
              evaluation_code_sha256=score_sources, comparison_hash=result["comparison_hash"],
              target_hash=compare["target_hash"], trainer=meta["training_config"]["backend"],
              hardware=hardware, chunk_rays=chunk, lpips=lpips_info, lpips_wheel_sha256=wheel_hash,
              alexnet_weights_sha256=weight_hash, test_views=len(rows), test_seconds=result["test_seconds"],
              view_psnr_ranks=rank_examples, mean=result["mean"],
              artifact_sha256={name: sha(path) for name, path in files.items()})
pending = review_zip.with_suffix(".tmp")
with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("stage4-review.json", json.dumps(review, indent=2, allow_nan=False))
    for name, path in files.items():
        archive.write(path, name)
with zipfile.ZipFile(pending) as archive:
    assert archive.testzip() is None and set(archive.namelist()) == set(files) | {"stage4-review.json"}
os.replace(pending, review_zip)
print(json.dumps(result["mean"], indent=2))
reference = result["dense_lego_reference"]
low, high = reference["range_db"]
print(f"Model {result['mean']['psnr']:.3f} dB; solid white {result['mean']['white_psnr']:.3f} dB.")
print(f"C4 reference {low:.2f}–{high:.2f} dB; inside range: {reference['inside_range']}.")
print(f"{len(rows)} held-out views in {result['test_seconds']/60:.1f} min.")
print(f"{review_zip.name} · {review_zip.stat().st_size:,} bytes · SHA-256 {sha(review_zip)}")

### Save the evaluation
Download `lego-dense-vanilla-s42-eval.zip` from Kaggle’s **Output** pane under `/kaggle/working`. The button below also embeds the archive in the notebook output. If your browser blocks the button, save a notebook version with outputs and download the ZIP from that version. Keep the step-50,000 checkpoint you already saved. Send the evaluation ZIP for review before starting the next training run.

In [ ]:
payload = base64.b64encode(review_zip.read_bytes()).decode("ascii")
display(HTML(f'<a download="{review_zip.name}" href="data:application/zip;base64,{payload}">Download {review_zip.name} (embedded copy)</a>'))